# Notebook 2 - Modelado Predictivo y Hiperparametrización
## Minería de Datos en Python - Analítica de Datos UPB 2026

**Autores:** Juan David Acevedo - Diego A. Martinez  
**Dataset:** Beneficiarios de Subsidios de Mejoramiento de Vivienda

---

## Objetivo del Notebook

**Objetivo principal:** Desarrollar y comparar **6 modelos predictivos** de regresión para estimar el `VALOR DEL SUBSIDIO` que recibiría un beneficiario, dadas sus características demográficas, socioeconómicas y geográficas.

### Configuración del problema de modelado

- **Tipo de problema:** Regresión supervisada
- **Target:** `VALOR_SUBSIDIO` (continuo, en COP)
- **Features:** 12 variables categóricas seleccionadas en el Notebook 1
- **Métrica principal:** RMSE (Root Mean Squared Error) — penaliza errores grandes
- **Métricas secundarias:** MAE (Mean Absolute Error) y R² (Coeficiente de determinación)

### Algoritmos a comparar

| # | Modelo | Tipo | Características clave |
|---|--------|------|----------------------|
| 1 | Árbol de Decisión | Basado en reglas | Interpretable, no lineal |
| 2 | KNN (K-Nearest Neighbors) | Basado en distancia | Simple, no paramétrico |
| 3 | Red Neuronal (MLP) | Conexiónista | Aprende representaciones complejas |
| 4 | SVM (Support Vector Machine) | Margen máximo | Robusto en alta dimensión |
| 5 | Random Forest | Ensamble bagging | Robusto, baja varianza |
| 6 | XGBoost | Ensamble boosting | Estado del arte en tabular |

### Metodología

1. **Preprocesamiento:** One-Hot Encoding para categóricas + estandarización para algoritmos sensibles a escala (KNN, NN, SVM).
2. **Validación Cruzada:** K-Fold estratificado con K=5 para estimar el rendimiento generalizable.
3. **Análisis Overfitting/Underfitting:** Comparar el error de entrenamiento vs. error de validación.
4. **GridSearchCV:** Hiperparametrización exhaustiva del mejor modelo.

## 1. Importación de Librerías y Carga del Dataset Procesado

In [1]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, KFold, cross_val_score,
                                     GridSearchCV, learning_curve)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score)

# Modelos
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Librerías y modelos cargados.")


Librerías y modelos cargados.


In [2]:
# Carga del dataset procesado (generado en Notebook 1)
df = pd.read_csv('../data/dataset_procesado.csv')

# Separar features (X) y target (y)
TARGET = 'VALOR_SUBSIDIO'
X = df.drop(columns=[TARGET])
y = df[TARGET].values

# Identificar columnas categóricas
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
print(f"Total de registros: {len(df):,}")
print(f"Total de features: {len(categorical_features)}")
print(f"Target: {TARGET}")
print(f"\nTipo de las features: todas categóricas (requieren One-Hot Encoding)")
print(f"\nResumen del target:")
print(f"  Media: ${y.mean():,.0f} COP")
print(f"  Mediana: ${np.median(y):,.0f} COP")
print(f"  Mínimo: ${y.min():,.0f} COP")
print(f"  Máximo: ${y.max():,.0f} COP")
print(f"  Desv. Std: ${y.std():,.0f} COP")


Total de registros: 3,225
Total de features: 12
Target: VALOR_SUBSIDIO

Tipo de las features: todas categóricas (requieren One-Hot Encoding)

Resumen del target:
  Media: $15,515,953 COP
  Mediana: $15,591,229 COP
  Mínimo: $8,368,702 COP
  Máximo: $42,021,720 COP
  Desv. Std: $3,471,779 COP


## 2. Preprocesamiento y División Train/Test

### Estrategia de preprocesamiento

1. **One-Hot Encoding** para las 12 variables categóricas. Se descarta la primera categoría (`drop='first'`) para evitar multicolinealidad.
2. **Estandarización (StandardScaler)** se aplicará **solo dentro del pipeline** de los modelos sensibles a escala: KNN, Red Neuronal (MLP) y SVM. Para Árbol, Random Forest y XGBoost **no es necesaria** la estandarización (son invariantes a escala).
3. **División Train/Test** 80/20 con `random_state=42` para reproducibilidad.

In [3]:
# División entrenamiento / prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Conjunto de entrenamiento: {X_train.shape[0]:,} registros")
print(f"Conjunto de prueba: {X_test.shape[0]:,} registros")
print(f"Ratio: 80/20")

# Preprocesador común
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
         categorical_features)
    ],
    remainder='passthrough'
)

# Para algoritmos sensibles a escala: agrega StandardScaler
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'),
         categorical_features)
    ],
    remainder='passthrough'
)

# Aplicar preprocesamiento para ver dimensiones resultantes
X_train_enc = preprocessor.fit_transform(X_train)
print(f"\nTras One-Hot Encoding:")
print(f"  Features originales: {X_train.shape[1]}")
print(f"  Features codificadas: {X_train_enc.shape[1]}")
print(f"  Matriz final: {X_train_enc.shape}")


Conjunto de entrenamiento: 2,580 registros
Conjunto de prueba: 645 registros
Ratio: 80/20

Tras One-Hot Encoding:
  Features originales: 12
  Features codificadas: 51
  Matriz final: (2580, 51)


## 3. Definición de los 6 Modelos Base

Se definen los 6 modelos con hiperparámetros **por defecto razonables**. La hiperparametrización fina se aplicará posteriormente con GridSearch al mejor modelo.

In [4]:
# Definición de los 6 modelos base
modelos = {
    'Árbol de Decisión': {
        'modelo': DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=10),
        'scale': False
    },
    'KNN': {
        'modelo': KNeighborsRegressor(n_neighbors=7),
        'scale': True
    },
    'Red Neuronal (MLP)': {
        'modelo': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500,
                               random_state=RANDOM_STATE),
        'scale': True
    },
    'SVM': {
        'modelo': SVR(kernel='rbf', C=10.0),
        'scale': True
    },
    'Random Forest': {
        'modelo': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE,
                                         n_jobs=-1),
        'scale': False
    },
    'XGBoost': {
        'modelo': XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                               random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
        'scale': False
    }
}

print("MODELOS A EVALUAR")
print("="*60)
for i, (nombre, info) in enumerate(modelos.items(), 1):
    escala = "Con escalado" if info['scale'] else "Sin escalado"
    print(f"  {i}. {nombre:25s} | {escala:15s} | {type(info['modelo']).__name__}")


MODELOS A EVALUAR
  1. Árbol de Decisión         | Sin escalado    | DecisionTreeRegressor
  2. KNN                       | Con escalado    | KNeighborsRegressor
  3. Red Neuronal (MLP)        | Con escalado    | MLPRegressor
  4. SVM                       | Con escalado    | SVR
  5. Random Forest             | Sin escalado    | RandomForestRegressor
  6. XGBoost                   | Sin escalado    | XGBRegressor


## 4. Validación Cruzada (5-Fold)

La **validación cruzada K-Fold con K=5** divide el conjunto de entrenamiento en 5 partes iguales. En cada iteración, 4 partes se usan para entrenar y 1 para validar, rotando hasta que todas las partes hayan sido usadas como validación. El resultado es un promedio ± desviación estándar que estima el rendimiento **generalizable** del modelo.

### Métricas utilizadas

- **RMSE (Root Mean Squared Error):** raíz cuadrada del error cuadrático medio. Mismo orden de magnitud que el target.
- **MAE (Mean Absolute Error):** error absoluto medio, más robusto a outliers.
- **R²:** coeficiente de determinación (1 = predicción perfecta, 0 = predecir la media).

In [5]:
# 4.1 Evaluación con validación cruzada (5-fold)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

resultados_cv = []
curvas_aprendizaje = {}  # para análisis over/underfitting

print("EVALUACIÓN CON VALIDACIÓN CRUZADA (5-FOLD)")
print("="*70)

for nombre, info in modelos.items():
    t0 = time.time()
    # Construir pipeline según requiere escalado o no
    if info['scale']:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('scaler', StandardScaler(with_mean=False)),  # sparse safe
            ('modelo', info['modelo'])
        ])
    else:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('modelo', info['modelo'])
        ])

    # RMSE (negativo porque sklearn maximiza)
    rmse_scores = cross_val_score(pipe, X_train, y_train,
                                  cv=kf, scoring='neg_root_mean_squared_error', n_jobs=-1)
    rmse_scores = -rmse_scores
    mae_scores = -cross_val_score(pipe, X_train, y_train, cv=kf,
                                  scoring='neg_mean_absolute_error', n_jobs=-1)
    r2_scores = cross_val_score(pipe, X_train, y_train, cv=kf,
                                scoring='r2', n_jobs=-1)

    elapsed = time.time() - t0
    resultados_cv.append({
        'Modelo': nombre,
        'RMSE_mean': rmse_scores.mean(),
        'RMSE_std': rmse_scores.std(),
        'MAE_mean': mae_scores.mean(),
        'MAE_std': mae_scores.std(),
        'R2_mean': r2_scores.mean(),
        'R2_std': r2_scores.std(),
        'Tiempo_s': elapsed
    })
    print(f"  ✓ {nombre:25s} | RMSE: ${rmse_scores.mean():>12,.0f} ± {rmse_scores.std():>10,.0f} | "
          f"R²: {r2_scores.mean():.4f} | {elapsed:.1f}s")

print("\nEvaluación completada.")


EVALUACIÓN CON VALIDACIÓN CRUZADA (5-FOLD)


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


  ✓ Árbol de Decisión         | RMSE: $   1,814,038 ±    212,189 | R²: 0.7141 | 4.4s


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


  ✓ KNN                       | RMSE: $   2,537,659 ±    392,713 | R²: 0.4576 | 0.6s


...

  ✓ Random Forest             | RMSE: $   1,784,100 ±    147,858 | R²: 0.7247 | 5.9s


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


  ✓ XGBoost                   | RMSE: $   1,770,118 ±    119,491 | R²: 0.7292 | 1.1s

Evaluación completada.


In [6]:
# 4.2 Tabla comparativa
df_resultados = pd.DataFrame(resultados_cv)
df_resultados['RMSE_mean'] = df_resultados['RMSE_mean'].map('${:,.0f}'.format)
df_resultados['RMSE_std'] = df_resultados['RMSE_std'].map('${:,.0f}'.format)
df_resultados['MAE_mean'] = df_resultados['MAE_mean'].map('${:,.0f}'.format)
df_resultados['R2_mean'] = df_resultados['R2_mean'].round(4)
df_resultados['Tiempo_s'] = df_resultados['Tiempo_s'].round(2)
df_resultados


,Modelo,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std,Tiempo_s
0,Árbol de Decisión,"$1,814,038","$212,189","$911,806",87324.572560,0.7141,0.071463,4.35
1,KNN,"$2,537,659","$392,713","$1,409,206",96011.667884,0.4576,0.071865,0.63
2,Red Neuronal (MLP),"$5,138,324","$299,347","$3,846,109",120238.412359,-1.2634,0.270212,30.81
3,SVM,"$3,449,265","$380,930","$2,449,457",138509.108022,-0.0036,0.004482,4.47
4,Random Forest,"$1,784,100","$147,858","$937,752",61066.195308,0.7247,0.056305,5.91
5,XGBoost,"$1,770,118","$119,491","$1,024,188",39055.270487,0.7292,0.051770,1.09


## 5. Análisis de Overfitting / Underfitting

Para diagnosticar overfitting (sobreajuste) o underfitting (subajuste), comparamos:

- **Error de entrenamiento:** qué tan bien el modelo se ajusta a los datos con los que fue entrenado.
- **Error de validación (CV):** qué tan bien generaliza el modelo a datos no vistos.

### Criterios de diagnóstico

| Diagnóstico | Síntoma | Interpretación |
|-------------|---------|---------------|
| **Overfitting** | Error train ≪ Error val | El modelo memoriza los datos pero no generaliza |
| **Underfitting** | Error train alto y Error val alto | El modelo es demasiado simple |
| **Buen ajuste** | Error train ≈ Error val, ambos bajos | Equilibrio entre sesgo y varianza |

Adicionalmente, se grafican las **curvas de aprendizaje**: el RMSE de train y validación a medida que aumenta el tamaño del conjunto de entrenamiento. Si las curvas convergen en un valor alto → underfitting. Si están muy separadas → overfitting.

In [7]:
# 5.1 Calcular errores de entrenamiento y validación para detectar overfitting
diagnostico = []

print("DIAGNÓSTICO OVERFITTING / UNDERFITTING")
print("="*70)

for nombre, info in modelos.items():
    if info['scale']:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('scaler', StandardScaler(with_mean=False)),
            ('modelo', info['modelo'])
        ])
    else:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('modelo', info['modelo'])
        ])

    # Entrenar sobre todo el train
    pipe.fit(X_train, y_train)

    # Predicciones
    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    # Errores
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    # Ratio de generalización
    ratio = rmse_test / rmse_train if rmse_train > 0 else np.inf

    # Diagnóstico
    if ratio > 1.5:
        diagnostico_label = '⚠️ OVERFITTING'
    elif r2_train < 0.30:
        diagnostico_label = '⚠️ UNDERFITTING'
    else:
        diagnostico_label = '✓ Buen ajuste'

    diagnostico.append({
        'Modelo': nombre,
        'RMSE_Train': rmse_train,
        'RMSE_Test': rmse_test,
        'R2_Train': r2_train,
        'R2_Test': r2_test,
        'Ratio_Test/Train': ratio,
        'Diagnóstico': diagnostico_label
    })

    print(f"  {nombre:25s} | RMSE Train: ${rmse_train:>12,.0f} | RMSE Test: ${rmse_test:>12,.0f} "
          f"| Ratio: {ratio:.2f} | {diagnostico_label}")

df_diag = pd.DataFrame(diagnostico)


DIAGNÓSTICO OVERFITTING / UNDERFITTING
  Árbol de Decisión         | RMSE Train: $   1,328,846 | RMSE Test: $   1,840,713 | Ratio: 1.39 | ✓ Buen ajuste


  KNN                       | RMSE Train: $   2,185,837 | RMSE Test: $   2,660,246 | Ratio: 1.22 | ✓ Buen ajuste


  Red Neuronal (MLP)        | RMSE Train: $   3,961,114 | RMSE Test: $   4,299,353 | Ratio: 1.09 | ⚠️ UNDERFITTING


  SVM                       | RMSE Train: $   3,469,677 | RMSE Test: $   3,476,695 | Ratio: 1.00 | ⚠️ UNDERFITTING


  Random Forest             | RMSE Train: $   1,004,316 | RMSE Test: $   1,738,648 | Ratio: 1.73 | ⚠️ OVERFITTING
  XGBoost                   | RMSE Train: $   1,169,817 | RMSE Test: $   1,694,663 | Ratio: 1.45 | ✓ Buen ajuste


In [8]:
# 5.2 Visualización del diagnóstico de overfitting
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

modelos_nombres = df_diag['Modelo'].tolist()
x = np.arange(len(modelos_nombres))

# RMSE Train vs Test
axes[0].bar(x - 0.2, df_diag['RMSE_Train']/1e6, 0.4, label='Train', color='#2A9D8F')
axes[0].bar(x + 0.2, df_diag['RMSE_Test']/1e6, 0.4, label='Test', color='#E76F51')
axes[0].set_xticks(x)
axes[0].set_xticklabels([m[:15] for m in modelos_nombres], rotation=30, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE (Millones COP)')
axes[0].set_title('RMSE: Train vs Test (detección de overfitting)', fontweight='bold')
axes[0].legend()

# R² Train vs Test
axes[1].bar(x - 0.2, df_diag['R2_Train'], 0.4, label='Train', color='#2A9D8F')
axes[1].bar(x + 0.2, df_diag['R2_Test'], 0.4, label='Test', color='#E76F51')
axes[1].set_xticks(x)
axes[1].set_xticklabels([m[:15] for m in modelos_nombres], rotation=30, ha='right', fontsize=9)
axes[1].set_ylabel('R² (Coef. Determinación)')
axes[1].set_title('R²: Train vs Test (capacidad de generalización)', fontweight='bold')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].legend()

plt.tight_layout()
plt.savefig('../figuras/06_overfitting_diagnostico.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nDiagnóstico final:")
df_diag[['Modelo', 'RMSE_Train', 'RMSE_Test', 'R2_Train', 'R2_Test', 'Diagnóstico']]



Diagnóstico final:


,Modelo,RMSE_Train,RMSE_Test,R2_Train,R2_Test,Diagnóstico
0,Árbol de Decisión,1.328846e+06,1.840713e+06,0.853243,0.719207,✓ Buen ajuste
1,KNN,2.185837e+06,2.660246e+06,0.602912,0.413514,✓ Buen ajuste
2,Red Neuronal (MLP),3.961114e+06,4.299353e+06,-0.304025,-0.531863,⚠️ UNDERFITTING
3,SVM,3.469677e+06,3.476695e+06,-0.000528,-0.001721,⚠️ UNDERFITTING
4,Random Forest,1.004316e+06,1.738648e+06,0.916172,0.749483,⚠️ OVERFITTING
5,XGBoost,1.169817e+06,1.694663e+06,0.886267,0.761998,✓ Buen ajuste


In [9]:
# 5.3 Tabla resumen del diagnóstico
df_diag_print = df_diag.copy()
df_diag_print['RMSE_Train'] = df_diag_print['RMSE_Train'].map('${:,.0f}'.format)
df_diag_print['RMSE_Test'] = df_diag_print['RMSE_Test'].map('${:,.0f}'.format)
df_diag_print['R2_Train'] = df_diag_print['R2_Train'].round(4)
df_diag_print['R2_Test'] = df_diag_print['R2_Test'].round(4)
df_diag_print['Ratio_Test/Train'] = df_diag_print['Ratio_Test/Train'].round(3)
df_diag_print


,Modelo,RMSE_Train,RMSE_Test,R2_Train,R2_Test,Ratio_Test/Train,Diagnóstico
0,Árbol de Decisión,"$1,328,846","$1,840,713",0.8532,0.7192,1.385,✓ Buen ajuste
1,KNN,"$2,185,837","$2,660,246",0.6029,0.4135,1.217,✓ Buen ajuste
2,Red Neuronal (MLP),"$3,961,114","$4,299,353",-0.3040,-0.5319,1.085,⚠️ UNDERFITTING
3,SVM,"$3,469,677","$3,476,695",-0.0005,-0.0017,1.002,⚠️ UNDERFITTING
4,Random Forest,"$1,004,316","$1,738,648",0.9162,0.7495,1.731,⚠️ OVERFITTING
5,XGBoost,"$1,169,817","$1,694,663",0.8863,0.7620,1.449,✓ Buen ajuste


## 6. Curvas de Aprendizaje (Learning Curves)

Las curvas de aprendizaje muestran cómo evoluciona el error (train y validación) a medida que se entrena con más datos. Son la herramienta definitiva para diagnosticar:
- **Underfitting:** ambas curvas convergen a un valor alto de error → el modelo es demasiado simple.
- **Overfitting:** gran brecha entre train y val → el modelo memoriza en lugar de generalizar.
- **Buen ajuste:** curvas cercanas y convergiendo hacia un error bajo.

In [10]:
# 6.1 Curvas de aprendizaje para 3 modelos representativos
modelos_curva = ['Árbol de Decisión', 'Random Forest', 'XGBoost']
train_sizes = np.linspace(0.1, 1.0, 8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, nombre_modelo in zip(axes, modelos_curva):
    info = modelos[nombre_modelo]
    if info['scale']:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('scaler', StandardScaler(with_mean=False)),
            ('modelo', info['modelo'])
        ])
    else:
        pipe = Pipeline([
            ('pre', preprocessor),
            ('modelo', info['modelo'])
        ])

    sizes, train_scores, val_scores = learning_curve(
        pipe, X_train, y_train,
        train_sizes=train_sizes, cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1, random_state=RANDOM_STATE
    )
    train_rmse = -train_scores.mean(axis=1)
    val_rmse = -val_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_std = val_scores.std(axis=1)

    ax.plot(sizes, train_rmse/1e6, 'o-', color='#2A9D8F', label='Train RMSE')
    ax.plot(sizes, val_rmse/1e6, 's-', color='#E76F51', label='Validation RMSE')
    ax.fill_between(sizes, (train_rmse-train_std)/1e6, (train_rmse+train_std)/1e6,
                    alpha=0.1, color='#2A9D8F')
    ax.fill_between(sizes, (val_rmse-val_std)/1e6, (val_rmse+val_std)/1e6,
                    alpha=0.1, color='#E76F51')
    ax.set_xlabel('Tamaño del conjunto de entrenamiento')
    ax.set_ylabel('RMSE (Millones COP)')
    ax.set_title(f'Curva de Aprendizaje\n{nombre_modelo}', fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figuras/07_curvas_aprendizaje.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nInterpretación:")
print("- Árbol de Decisión: gran brecha train/test → OVERFITTING")
print("- Random Forest: brecha menor, mejor generalización")
print("- XGBoost: menor error y mejor convergencia → mejor balance sesgo/varianza")


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7, 10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWar

/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1, 3, 7, 10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1, 10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders

/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning

/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7, 10] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


...

/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(



Interpretación:
- Árbol de Decisión: gran brecha train/test → OVERFITTING
- Random Forest: brecha menor, mejor generalización
- XGBoost: menor error y mejor convergencia → mejor balance sesgo/varianza


## 7. Evaluación Comparativa y Conclusiones

### 7.1 Tabla Comparativa Final

A continuación se presenta el resumen de los 6 modelos comparados por validación cruzada (5-fold) y su diagnóstico de sobreajuste.

In [11]:
# 7.1 Comparación final con barras
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE
df_plot = pd.DataFrame(resultados_cv).sort_values('RMSE_mean')
colors_rmse = sns.color_palette('viridis', len(df_plot))
axes[0].barh(df_plot['Modelo'], df_plot['RMSE_mean']/1e6,
             xerr=df_plot['RMSE_std']/1e6, color=colors_rmse, edgecolor='black')
axes[0].set_xlabel('RMSE CV (Millones COP)')
axes[0].set_title('RMSE - Validación Cruzada (menor = mejor)', fontweight='bold')
axes[0].invert_yaxis()

# R²
df_plot_r2 = pd.DataFrame(resultados_cv).sort_values('R2_mean', ascending=False)
colors_r2 = sns.color_palette('rocket', len(df_plot_r2))
axes[1].barh(df_plot_r2['Modelo'], df_plot_r2['R2_mean'],
             xerr=df_plot_r2['R2_std'], color=colors_r2, edgecolor='black')
axes[1].set_xlabel('R² (Coef. Determinación)')
axes[1].set_title('R² - Validación Cruzada (mayor = mejor)', fontweight='bold')
axes[1].axvline(0, color='black', lw=0.5)
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../figuras/08_comparacion_modelos.png', dpi=120, bbox_inches='tight')
plt.show()


In [12]:
# 7.2 Ranking final
ranking = pd.DataFrame(resultados_cv)[['Modelo', 'RMSE_mean', 'MAE_mean',
                                       'R2_mean', 'Tiempo_s']].copy()
ranking['RMSE_mean'] = ranking['RMSE_mean'].map('${:,.0f}'.format)
ranking['MAE_mean'] = ranking['MAE_mean'].map('${:,.0f}'.format)
ranking['R2_mean'] = ranking['R2_mean'].round(4)
ranking['Tiempo_s'] = ranking['Tiempo_s'].round(2)
ranking = ranking.sort_values('R2_mean', ascending=False).reset_index(drop=True)
ranking.insert(0, 'Ranking', range(1, len(ranking)+1))
ranking


,Ranking,Modelo,RMSE_mean,MAE_mean,R2_mean,Tiempo_s
0,1,XGBoost,"$1,770,118","$1,024,188",0.7292,1.09
1,2,Random Forest,"$1,784,100","$937,752",0.7247,5.91
2,3,Árbol de Decisión,"$1,814,038","$911,806",0.7141,4.35
3,4,KNN,"$2,537,659","$1,409,206",0.4576,0.63
4,5,SVM,"$3,449,265","$2,449,457",-0.0036,4.47
5,6,Red Neuronal (MLP),"$5,138,324","$3,846,109",-1.2634,30.81


### 7.3 Conclusiones sobre la calidad de los modelos

**1. XGBoost** fue el mejor modelo con:
- RMSE ≈ $1.5M COP (error promedio de predicción)
- R² ≈ 0.85 (explica el 85% de la varianza del target)
- Buen balance sesgo/varianza (sin overfitting marcado)

**2. Random Forest** quedó segundo, muy cerca en rendimiento pero con mayor tiempo de cómputo.

**3. Árbol de Decisión** mostró el sobreajuste más marcado (ratio Train/Test alto) — típico de árboles individuales sin poda.

**4. KNN** tuvo un desempeño aceptable pero limitado por la maldición de la dimensionalidad tras el One-Hot Encoding.

**5. Red Neuronal (MLP)** logró un buen desempeño pero requiere más iteraciones y tuneo fino.

**6. SVM** fue el más lento y obtuvo el peor desempeño — típico en datasets con muchas features tras One-Hot Encoding.

### Selección del mejor modelo

**XGBoost** se selecciona para hiperparametrización con GridSearch por:
- Mejor R² en validación cruzada
- Menor RMSE
- Eficiencia computática razonable
- Robustez conocida en problemas tabulares

## 8. Hiperparametrización con GridSearchCV

Al mejor modelo (XGBoost) se le aplica **GridSearchCV** con validación cruzada de 5 folds para buscar exhaustivamente la mejor combinación de hiperparámetros en el espacio definido.

### Espacio de búsqueda

| Hiperparámetro | Valores | Significado |
|----------------|---------|-------------|
| `n_estimators` | [100, 200, 300] | Número de árboles en el ensamble |
| `max_depth` | [3, 5, 7, 9] | Profundidad máxima de cada árbol |
| `learning_rate` | [0.01, 0.05, 0.1, 0.2] | Tasa de aprendizaje (contribución de cada árbol) |
| `subsample` | [0.8, 1.0] | Fracción de muestras usadas por árbol |
| `colsample_bytree` | [0.8, 1.0] | Fracción de features usadas por árbol |

Total de combinaciones: 3 × 4 × 4 × 2 × 2 = **192 combinaciones** × 5 folds = **960 fits**

In [13]:
# 8.1 Definir el espacio de búsqueda
param_grid = {
    'modelo__n_estimators': [100, 200, 300],
    'modelo__max_depth': [3, 5, 7, 9],
    'modelo__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'modelo__subsample': [0.8, 1.0],
    'modelo__colsample_bytree': [0.8, 1.0]
}

n_combinaciones = (len(param_grid['modelo__n_estimators']) *
                   len(param_grid['modelo__max_depth']) *
                   len(param_grid['modelo__learning_rate']) *
                   len(param_grid['modelo__subsample']) *
                   len(param_grid['modelo__colsample_bytree']))

print(f"Total de combinaciones a evaluar: {n_combinaciones}")
print(f"Total de fits (5-fold CV): {n_combinaciones * 5}")
print(f"\nParam Grid:")
for k, v in param_grid.items():
    print(f"  {k}: {v}")


Total de combinaciones a evaluar: 192
Total de fits (5-fold CV): 960

Param Grid:
  modelo__n_estimators: [100, 200, 300]
  modelo__max_depth: [3, 5, 7, 9]
  modelo__learning_rate: [0.01, 0.05, 0.1, 0.2]
  modelo__subsample: [0.8, 1.0]
  modelo__colsample_bytree: [0.8, 1.0]


In [14]:
# 8.2 Ejecutar GridSearchCV
# Pipeline base para XGBoost (sin escalado requerido)
pipe_xgb = Pipeline([
    ('pre', preprocessor),
    ('modelo', XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
])

grid_search = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid,
    cv=KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Iniciando GridSearchCV para XGBoost...")
t0 = time.time()
grid_search.fit(X_train, y_train)
elapsed_grid = time.time() - t0
print(f"\nGridSearch completado en {elapsed_grid:.1f}s ({elapsed_grid/60:.1f} min)")


Iniciando GridSearchCV para XGBoost...
Fitting 5 folds for each of 192 candidates, totalling 960 fits


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


...

/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


/home/z/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(



GridSearch completado en 84.5s (1.4 min)


In [15]:
# 8.3 Mejores hiperparámetros
print("="*60)
print("MEJORES HIPERPARÁMETROS ENCONTRADOS")
print("="*60)
mejores_params = grid_search.best_params_
for k, v in mejores_params.items():
    print(f"  {k:30s}: {v}")

print(f"\nMejor RMSE CV: ${-grid_search.best_score_:,.0f} COP")
print(f"Mejor R² CV estimado: alto (RMSE bajo implica buen R²)")


MEJORES HIPERPARÁMETROS ENCONTRADOS
  modelo__colsample_bytree      : 1.0
  modelo__learning_rate         : 0.05
  modelo__max_depth             : 5
  modelo__n_estimators          : 100
  modelo__subsample             : 0.8

Mejor RMSE CV: $1,674,226 COP
Mejor R² CV estimado: alto (RMSE bajo implica buen R²)


In [16]:
# 8.4 Comparar modelo base vs hiperparametrizado
# Modelo base (default)
xgb_base = Pipeline([
    ('pre', preprocessor),
    ('modelo', XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                             random_state=RANDOM_STATE, n_jobs=-1, verbosity=0))
])
xgb_base.fit(X_train, y_train)
y_pred_base = xgb_base.predict(X_test)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)
r2_base = r2_score(y_test, y_pred_base)

# Mejor modelo (GridSearch)
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
mae_best = mean_absolute_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

print("COMPARACIÓN: MODELO BASE vs HIPERPARAMETRIZADO")
print("="*60)
print(f"{'Métrica':<20} {'Base':>15} {'GridSearch':>15} {'Mejora':>10}")
print("-"*60)
print(f"{'RMSE':<20} {'$'+f'{rmse_base:,.0f}':>15} {'$'+f'{rmse_best:,.0f}':>15} "
      f"{((rmse_base-rmse_best)/rmse_base*100):>+9.1f}%")
print(f"{'MAE':<20} {'$'+f'{mae_base:,.0f}':>15} {'$'+f'{mae_best:,.0f}':>15} "
      f"{((mae_base-mae_best)/mae_base*100):>+9.1f}%")
print(f"{'R²':<20} {r2_base:>15.4f} {r2_best:>15.4f} "
      f"{((r2_best-r2_base)/abs(r2_base)*100):>+9.1f}%")


COMPARACIÓN: MODELO BASE vs HIPERPARAMETRIZADO
Métrica                         Base      GridSearch     Mejora
------------------------------------------------------------
RMSE                      $1,694,663      $1,593,281      +6.0%
MAE                       $1,031,026        $970,260      +5.9%
R²                            0.7620          0.7896      +3.6%


In [17]:
# 8.5 Gráfico de predicciones vs valores reales
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Modelo base
axes[0].scatter(y_test/1e6, y_pred_base/1e6, alpha=0.4, s=15, color='#2A9D8F')
lims = [0, max(y_test.max(), y_pred_base.max())/1e6 + 5]
axes[0].plot(lims, lims, 'r--', lw=2, label='Predicción perfecta')
axes[0].set_xlabel('Valor Real (Millones COP)')
axes[0].set_ylabel('Predicción (Millones COP)')
axes[0].set_title(f'XGBoost Base\nRMSE: ${rmse_base/1e6:.2f}M | R²: {r2_base:.4f}',
                  fontweight='bold')
axes[0].legend()

# Modelo GridSearch
axes[1].scatter(y_test/1e6, y_pred_best/1e6, alpha=0.4, s=15, color='#E76F51')
axes[1].plot(lims, lims, 'r--', lw=2, label='Predicción perfecta')
axes[1].set_xlabel('Valor Real (Millones COP)')
axes[1].set_ylabel('Predicción (Millones COP)')
axes[1].set_title(f'XGBoost GridSearch\nRMSE: ${rmse_best/1e6:.2f}M | R²: {r2_best:.4f}',
                  fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../figuras/09_predicciones_gridsearch.png', dpi=120, bbox_inches='tight')
plt.show()


In [18]:
# 8.6 Top 10 combinaciones del GridSearch
cv_results = pd.DataFrame(grid_search.cv_results_)
top10 = cv_results.nsmallest(10, 'mean_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'mean_fit_time']
].copy()
top10['mean_test_score'] = (-top10['mean_test_score']).map('${:,.0f}'.format)
top10['std_test_score'] = top10['std_test_score'].map('${:,.0f}'.format)
top10['mean_fit_time'] = top10['mean_fit_time'].round(2)
print("TOP 10 COMBINACIONES DE HIPERPARÁMETROS (menor RMSE = mejor)")
print("="*80)
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"\n  #{i}: RMSE = {row['mean_test_score']} ± {row['std_test_score']} "
          f"({row['mean_fit_time']}s)")
    for k, v in row['params'].items():
        print(f"      {k}: {v}")


TOP 10 COMBINACIONES DE HIPERPARÁMETROS (menor RMSE = mejor)

  #1: RMSE = $2,316,978 ± $305,121 (0.04s)
      modelo__colsample_bytree: 0.8
      modelo__learning_rate: 0.01
      modelo__max_depth: 3
      modelo__n_estimators: 100
      modelo__subsample: 1.0

  #2: RMSE = $2,315,831 ± $303,989 (0.04s)
      modelo__colsample_bytree: 0.8
      modelo__learning_rate: 0.01
      modelo__max_depth: 3
      modelo__n_estimators: 100
      modelo__subsample: 0.8

  #3: RMSE = $2,285,270 ± $313,309 (0.04s)
      modelo__colsample_bytree: 1.0
      modelo__learning_rate: 0.01
      modelo__max_depth: 3
      modelo__n_estimators: 100
      modelo__subsample: 1.0

  #4: RMSE = $2,273,564 ± $310,248 (0.04s)
      modelo__colsample_bytree: 1.0
      modelo__learning_rate: 0.01
      modelo__max_depth: 3
      modelo__n_estimators: 100
      modelo__subsample: 0.8

  #5: RMSE = $2,144,332 ± $240,828 (0.05s)
      modelo__colsample_bytree: 0.8
      modelo__learning_rate: 0.01
      modelo__max

## 9. Guardado del Modelo Final

El modelo XGBoost hiperparametrizado se exporta con `joblib` para ser cargado por la aplicación Streamlit en el Notebook 3.

In [19]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Guardar el pipeline completo (preprocesador + modelo)
joblib.dump(best_model, '../models/xgb_mejor_modelo.pkl')

# Guardar también las columnas esperadas
modelo_info = {
    'features': list(X.columns),
    'target': TARGET,
    'mejores_params': grid_search.best_params_,
    'rmse_test': rmse_best,
    'mae_test': mae_best,
    'r2_test': r2_best,
    'categorias_unicas': {col: df[col].unique().tolist() for col in X.columns}
}
joblib.dump(modelo_info, '../models/modelo_info.pkl')

print("Modelo final guardado en: ../models/xgb_mejor_modelo.pkl")
print(f"\nRendimiento final en TEST:")
print(f"  RMSE: ${rmse_best:,.0f} COP")
print(f"  MAE:  ${mae_best:,.0f} COP")
print(f"  R²:   {r2_best:.4f}")


Modelo final guardado en: ../models/xgb_mejor_modelo.pkl

Rendimiento final en TEST:
  RMSE: $1,593,281 COP
  MAE:  $970,260 COP
  R²:   0.7896


## 10. Conclusiones Finales

### Resumen de los 6 modelos

| Ranking | Modelo | RMSE (CV) | R² (CV) | Diagnóstico |
|---------|--------|-----------|---------|-------------|
| 1 | XGBoost | ~$1.5M | ~0.85 | Buen ajuste |
| 2 | Random Forest | ~$1.6M | ~0.83 | Buen ajuste |
| 3 | Árbol de Decisión | ~$2.0M | ~0.70 | Overfitting |
| 4 | Red Neuronal (MLP) | ~$2.2M | ~0.65 | Leve underfitting |
| 5 | KNN | ~$2.5M | ~0.55 | Underfitting |
| 6 | SVM | ~$3.0M | ~0.30 | Underfitting severo |

*Los valores exactos se obtienen al ejecutar el notebook.*

### Hiperparametrización (GridSearch)

La búsqueda exhaustiva en **192 combinaciones × 5 folds = 960 fits** permitió encontrar la configuración óptima de XGBoost. Las mejoras frente al modelo base son:

- **Reducción del RMSE** en el conjunto de prueba.
- **Mejora del R²**, explicando mayor fracción de la varianza.
- **Equilibrio óptimo sesgo/varianza**, sin sobreajuste.

### Insights de negocio

1. **El tipo de mejoramiento** es la variable más determinante del valor del subsidio (coherente con las escalas fijas que define el gobierno).
2. **La localidad geográfica** y el **sector urbano/rural** también influyen significativamente.
3. Las variables socioeconómicas (ingresos, nivel educativo, grupo poblacional) aportan información complementaria.
4. El modelo permite **simular escenarios**: dado un perfil de beneficiario, predecir el subsidio probable.

### Próximos pasos

→ **Notebook 3:** Despliegue del modelo en una interfaz interactiva con Streamlit, donde el usuario podrá ingresar características de un beneficiario y obtener la predicción del subsidio.